In [20]:
import pandas as pd

In [21]:
df=pd.read_csv('../../datasets/Car Dataset Processed.csv')

In [22]:
df.isna().sum()

Unnamed: 0            0
car_name              0
registration_year     0
insurance_validity    0
fuel_type             0
seats                 0
kms_driven            0
ownsership            0
transmission          0
manufacturing_year    0
mileage(kmpl)         0
engine(cc)            0
max_power(bhp)        0
torque(Nm)            0
price(in lakhs)       0
dtype: int64

In [23]:
df['brand'] = df['car_name'].str.split().str[1]
df.drop('car_name', axis=1, inplace=True)

In [24]:
df.drop_duplicates()

,Unnamed: 0,registration_year,insurance_validity,fuel_type,seats,kms_driven,ownsership,transmission,manufacturing_year,mileage(kmpl),engine(cc),max_power(bhp),torque(Nm),price(in lakhs),brand
0,0,Jul-17,Comprehensive,Petrol,5,56000,First Owner,Automatic,2017,7.81,2996.0,2996.0,333.0,63.75,Mercedes-Benz
1,1,Jan-21,Comprehensive,Petrol,5,30615,First Owner,Automatic,2020,17.40,999.0,999.0,9863.0,8.99,Nissan
2,2,Sep-18,Comprehensive,Diesel,5,24000,First Owner,Automatic,2018,20.68,1995.0,1995.0,188.0,23.75,BMW
3,3,Dec-19,Comprehensive,Petrol,5,18378,First Owner,Manual,2019,16.50,1353.0,1353.0,13808.0,13.56,Kia
4,4,Aug-19,Comprehensive,Petrol,5,44900,First Owner,Automatic,2019,14.67,1798.0,1798.0,17746.0,24.00,Skoda
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1494,1547,May-22,Third Party insurance,CNG,7,26500,First Owner,Manual,2022,26.11,1462.0,1462.0,8663.0,11.50,Maruti
1495,1548,Aug-20,Comprehensive,Diesel,5,35000,First Owner,Automatic,2020,1493.00,11345.0,11345.0,250.0,17.41,Hyundai
1496,1550,Jun-17,Comprehensive,Petrol,5,49000,First Owner,Manual,2017,17.50,1199.0,1199.0,887.0,5.85,Honda
1497,1551,May-18,Comprehensive,Petrol,5,40000,Second Owner,Manual,2018,18.78,999.0,999.0,75.0,4.75,Volkswagen


In [25]:
numeric_columns = ['seats','kms_driven','manufacturing_year','mileage(kmpl)','engine(cc)','max_power(bhp)','torque(Nm)','price(in lakhs)']

In [26]:
df = df[
    (df['price(in lakhs)'] > 0) &
    (df['price(in lakhs)'] < 100)]

df = df[
    (df['seats'] >= 2) &
    (df['seats'] <= 10)
]

df = df[
    (df['mileage(kmpl)'] > 0) &
  (df['mileage(kmpl)'] < 100)
]

df = df[
    (df['engine(cc)'] > 500) &
    (df['engine(cc)'] < 10000)
]

df = df[
    (df['max_power(bhp)'] > 10) &
    (df['max_power(bhp)'] < 2000)
]

df = df[
    (df['torque(Nm)'] > 10) &
    (df['torque(Nm)'] < 2000)
]

df = df.reset_index(drop=True)

In [27]:
X = df[
    [ 'fuel_type',
        'seats',
        'kms_driven',
        'ownsership',
        'transmission',
        'manufacturing_year',
        'mileage(kmpl)',
        'engine(cc)',
        'max_power(bhp)',
        'torque(Nm)'
    ]
]
y = df['price(in lakhs)']

In [28]:
X = pd.get_dummies(
    X,
    columns=[
        'fuel_type',
        'ownsership',
        'transmission'
    ],
    drop_first=True,
    dtype=int
)

In [29]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_cols = ['fuel_type', 'transmission']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

In [30]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=48)

In [31]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
scaler.fit(X_train) 
X_train_scaled=scaler.transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [32]:
from sklearn.neighbors import KNeighborsRegressor
model=KNeighborsRegressor(n_neighbors=1)
model.fit(X_train_scaled,y_train)
model.score(X_test_scaled,y_test)

0.9783710150900795

In [33]:
from sklearn.metrics import r2_score

In [34]:
ya = y_test
yp = model.predict(X_test_scaled)

In [35]:
r2_score(ya,yp)

0.9783710150900795

In [17]:
import pickle 
pickle.dump(model,open('carprice_charges.pkl','wb'))

In [18]:
print(X.columns.tolist())

['seats', 'kms_driven', 'manufacturing_year', 'mileage(kmpl)', 'engine(cc)', 'max_power(bhp)', 'torque(Nm)', 'fuel_type_Diesel', 'fuel_type_Petrol', 'ownsership_Second Owner', 'ownsership_Third Owner', 'transmission_Manual']


In [19]:
pickle.dump(scaler,open('scaler.pkl','wb'))